In [1]:
import numpy as np
np.random.seed(1337) 
import pandas as pd
import keras
import keras.regularizers
import glob
import os
import itertools
import seaborn as sns


from sklearn.preprocessing import MinMaxScaler, StandardScaler, MaxAbsScaler, Normalizer
from sklearn.metrics import confusion_matrix, recall_score, f1_score, precision_score

from keras_self_attention import SeqSelfAttention
from keras.utils import custom_object_scope

import matplotlib.pyplot as plt

In [2]:
model_loc = '/Users/dimitrygrebenyuk/Yandex.Disk.localized/Working/PDF/Refinements/PDF-Cluster-Prediction/all_data/cifs/new_fold_3_model.hdf5'

with custom_object_scope({'SeqSelfAttention': SeqSelfAttention}):
    load_model = keras.models.load_model(model_loc)

os.chdir('/Users/dimitrygrebenyuk/Yandex.Disk.localized/Working/PDF/Refinements/PDF-Cluster-Prediction/experimentalPDF')
files = glob.glob('*processed.gr')

In [3]:
raw_data_points = []

with open('labels.txt', 'w') as labels:
    for f in files:
        df = pd.read_csv(f, usecols=[1], skiprows=1, header=None, delim_whitespace=True, skipfooter=1, engine='python')

        raw_data_points.append(df.values.ravel())
        if f[0] == 'p':
            labels.write('10')
            labels.write('\n')
        else:    
            labels.write(f[0])
            labels.write('\n')
raw_data_points = np.array(raw_data_points)

normalize = Normalizer()
data_points = normalize.fit_transform(raw_data_points)

# Load the labels
labels = pd.read_csv("labels.txt", header=None)
labels = labels.values.ravel()  # convert the labels to a 1D array

In [4]:
load_model.evaluate(data_points, labels)
y_pred_prob = load_model.predict(data_points)
y_pred = np.argmax(y_pred_prob, axis=1)
confusion = confusion_matrix(labels, y_pred)
recall = recall_score(labels, y_pred, average=None)
f1 = f1_score(labels, y_pred, average=None)
precision = precision_score(labels, y_pred, average=None)
print(np.unique(labels), np.unique(y_pred))
print('Confusion matrix \n', confusion)
print('Recall score:', recall)
print('F1 score:', f1)
print('Precision score:', precision)

1/1 [==============================] - 0s 129ms/step - loss: 0.2436 - accuracy: 1.0000


2023-09-12 09:39:30.041844: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


1/1 [==============================] - 0s 84ms/step
[ 4  6 10] [ 4  6 10]
Confusion matrix 
 [[2 0 0]
 [0 2 0]
 [0 0 1]]
Recall score: [1. 1. 1.]
F1 score: [1. 1. 1.]
Precision score: [1. 1. 1.]
